In [1]:
import warnings
warnings.filterwarnings("ignore")

# Device Check

In [2]:
import torch

print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
NVIDIA GeForce RTX 4070


# Configs

In [3]:
TICKERS = [
    "AAPL","AXP","KO","BAC","CVX",
    "MCO","OXY","CB","KHC","GOOGL"
]

START_DATE = "2021-01-01"
END_DATE = "2026-06-01"

FORECAST_HORIZON = 120  # ~6 months (20 * 6)
TOP_K = 5

# Download Dataset

In [4]:
import yfinance as yf
print("Downloading data...")

raw = yf.download(
    TICKERS,
    start=START_DATE,
    end=END_DATE,
    auto_adjust=True
)

[*********************100%***********************]  10 of 10 completed


# Feature Extraction

In [5]:
import pandas as pd
from ta.trend import SMAIndicator, EMAIndicator, MACD
from ta.momentum import RSIIndicator
from ta.volatility import BollingerBands

def create_features(df):
    df = df.copy()

    # SMA
    df["sma20"] = SMAIndicator(df["Close"], window=20).sma_indicator()
    df["sma50"] = SMAIndicator(df["Close"], window=50).sma_indicator()

    # EMA
    df["ema20"] = EMAIndicator(df["Close"], window=20).ema_indicator()

    # RSI
    df["rsi"] = RSIIndicator(df["Close"], window=14).rsi()

    # MACD
    macd = MACD(df["Close"])
    df["macd"] = macd.macd()
    df["macd_signal"] = macd.macd_signal()

    # Bollinger Bands
    bb = BollingerBands(df["Close"])
    df["bb_high"] = bb.bollinger_hband()
    df["bb_low"] = bb.bollinger_lband()

    # Returns
    df["returns"] = df["Close"].pct_change()

    # Volatility
    df["volatility"] = df["returns"].rolling(20).std()

    df = df.dropna()

    return df

# Training and Evaluation

## Evaluation Dstat Function

In [6]:
def directional_accuracy(y_true, y_pred):
    actual_direction = np.sign(np.diff(y_true))
    pred_direction = np.sign(np.diff(y_pred))

    correct = (actual_direction == pred_direction).sum()
    total = len(actual_direction)

    return correct / total

## Training PatchTHT and Evaluation with Data Test

In [17]:
models = "NBEATSx"

In [18]:
EXPORT_FORECAST_REPORT_PLOT = f"./reports/forecast_report_{models}.pdf"

In [19]:
from neuralforecast import NeuralForecast
from neuralforecast.models import PatchTST
from neuralforecast.models import NBEATSx
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.preprocessing import StandardScaler
from matplotlib.backends.backend_pdf import PdfPages
from neuralforecast.models import TFT

all_forecasts = []
plot_pdf = PdfPages(EXPORT_FORECAST_REPORT_PLOT)
for ticker in TICKERS:
    df = pd.DataFrame({
        "Date": raw.index,
        "Close": raw["Close"][ticker].values,
        "Volume": raw["Volume"][ticker].values
    })
    df.to_csv(f"datasets/{ticker}-raw.csv", index=False)
    df = create_features(df)
    df.to_csv(f"datasets/{ticker}-indicator.csv", index=False)
    
    nf_df = pd.DataFrame({
        "unique_id": ticker,
        "ds": pd.to_datetime(df["Date"]),
        "y": df["Close"],

        # Other features
        "volume": df["Volume"],
        "sma20": df["sma20"],
        "sma50": df["sma50"],
        "ema20": df["ema20"],
        "rsi": df["rsi"],
        "macd": df["macd"],
        "macd_signal": df["macd_signal"],
        "bb_low": df["bb_low"],
        "bb_high": df["bb_high"],
        "returns": df["returns"],
        "volatility": df["volatility"],
    })

    train_size = len(nf_df) - FORECAST_HORIZON
    
    train_df = nf_df.iloc[:train_size]
    test_df = nf_df.iloc[train_size:]

    model = None
    if models == "PatchTST":
        model = PatchTST(
            h=FORECAST_HORIZON,
            input_size=504,
            patch_len=16,
            stride=8,
            hidden_size=128,
            n_heads=8,
            learning_rate=1e-3,
            max_steps=500,
            batch_size=32
        )
    
    if models == "TFT":
        model = TFT(
            h=FORECAST_HORIZON,
            input_size=252,
            hidden_size=64,
            n_head=4,
            learning_rate=1e-3,
            max_steps=100,
            batch_size=32,
            hist_exog_list=[
                "volume",
                "sma20",
                "sma50",
                "ema20",
                "rsi",
                "macd",
                "macd_signal",
                "bb_low",
                "bb_high",
                "returns",
                "volatility",
            ]
        )
        
    if models == "NBEATSx":
         model = NBEATSx(
            h=FORECAST_HORIZON,
            input_size=252,
        
            hist_exog_list=[
                "volume",
                "sma20",
                "sma50",
                "ema20",
                "rsi",
                "macd",
                "macd_signal",
                "bb_low",
                "bb_high",
                "returns",
                "volatility",
            ],
        
            learning_rate=1e-3,
            max_steps=300,
            batch_size=32,
        )

    

    nf = NeuralForecast(
        models=[model],
        freq='D'
    )

    # Train
    nf.fit(df=train_df)

    forecast = nf.predict()
    preds = forecast[models].values

    # evaluation
    y_true = test_df["y"].values[:len(preds)]
    mae = mean_absolute_error(y_true, preds)
    rmse = np.sqrt(mean_squared_error(y_true, preds))
    dstat = directional_accuracy(y_true, preds)
    print(f"Dstat {ticker}: {dstat}")

    # calculation
    current_price = df["Close"].iloc[-1]
    forecast_price = preds[-1]
    volatility = df["volatility"].iloc[-1]
    expected_return = ((forecast_price - current_price) / current_price) * 100
    all_forecasts.append({
        "Ticker": ticker,
        "CurrentPrice": current_price,
        "ForecastPrice": forecast_price,
        "ExpectedReturn": expected_return,
        "MAE": mae,
        "RMSE": rmse,
        "Dstat": dstat,
        "Volatility": volatility
    })


    plt.figure(figsize=(12,6))
    plt.plot(test_df["ds"][:len(preds)], y_true, label="Actual")
    plt.plot(test_df["ds"][:len(preds)], preds, label="Forecast")

    plt.title(f"{ticker} Forecast")
    plt.xlabel("Date")
    plt.ylabel("Price")
    plt.legend()
    plt.grid(True)

    plot_pdf.savefig()
    plt.close()
plot_pdf.close()

Seed set to 1
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name         | Type          | Params | Mode 
-------------------------------------------------------
0 | loss         | MAE           | 0      | train
1 | padder_train | ConstantPad1d | 0      | train
2 | scaler       | TemporalNorm  | 0      | train
3 | blocks       | ModuleList    | 7.5 M  | train
-------------------------------------------------------
7.4 M     Trainable params
89.7 K    Non-trainable params
7.5 M     Total params
30.152    Total estimated model params size (MB)
31        Modules in train mode
0         Modules in eval mode


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_steps=300` reached.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Predicting: |          | 0/? [00:00<?, ?it/s]

Dstat GOOGL: 0.5126050420168067


## Export Stock Forecast Report and Get Top 5 Stock Prediction

In [20]:
EXPORT_FORECAST_PORTOFOLIO = f"./reports/all_stock_forecast_{models}.csv"

In [21]:
result_df = pd.DataFrame(all_forecasts)
result_df = result_df.sort_values(
    by="ExpectedReturn",
    ascending=False
)

result_df.to_csv(EXPORT_FORECAST_PORTOFOLIO, index=False)
print(f"{EXPORT_FORECAST_PORTOFOLIO} saved.")

./reports/all_stock_forecast_NBEATSx.csv saved.


# Retrain the model for 6-month-ahead forecasting

## Get Best Model from PatchTST/TFT/NBEATSx

In [22]:
import pandas as pd

files = [
    "all_stock_forecast_PatchTST.csv",
    "all_stock_forecast_TFT.csv",
    "all_stock_forecast_NBEATSx.csv"
]

summary = []

for file in files:
    df = pd.read_csv(f"./reports/{file}")

    summary.append({
        "Model": file.replace("all_stock_forecast_", "").replace(".csv", ""),
        "Avg_Dstat": df["Dstat"].mean(),
        "Avg_MAE": df["MAE"].mean(),
        "Avg_RMSE": df["RMSE"].mean(),
    })

summary_df = pd.DataFrame(summary)
summary_df.sort_values(
    by=["Avg_Dstat", "Avg_RMSE"],
    ascending=[False, True]
)

print(summary_df)

best_model = summary_df.iloc[0]

print("\n====================")
print("BEST MODEL")
print("====================")
print(best_model["Model"])

      Model  Avg_Dstat     Avg_MAE    Avg_RMSE
0  PatchTST   0.510924   30.331319   36.163167
1       TFT   0.494958   20.794641   24.590278
2   NBEATSx   0.505042  293.641392  337.905940

BEST MODEL
PatchTST


## ReTrain model with all stocks from previous train

In [23]:
BEST_MODEL="PatchTST"
FORECAST_TESTING_REPORT=f"./reports/all_stock_forecast_{BEST_MODEL}.csv"

In [24]:
import pandas as pd

df_stock = pd.read_csv(FORECAST_TESTING_REPORT)
tickers = df_stock["Ticker"].tolist()

In [26]:
results = []
for ticker in tickers:
    df = pd.DataFrame({
        "Date": raw.index,
        "Close": raw["Close"][ticker].values,
        "Volume": raw["Volume"][ticker].values
    })

    df = create_features(df)
    nf_df = pd.DataFrame({
        "unique_id": ticker,
        "ds": pd.to_datetime(df["Date"]),
        "y": df["Close"],
        
        "volume": df["Volume"],
        "sma20": df["sma20"],
        "sma50": df["sma50"],
        "ema20": df["ema20"],
        "rsi": df["rsi"],
        "macd": df["macd"],
        "macd_signal": df["macd_signal"],
        "bb_low": df["bb_low"],
        "bb_high": df["bb_high"],
        "returns": df["returns"],
        "volatility": df["volatility"],
    })

    model = None
    if BEST_MODEL == "PatchTST":
        model = PatchTST(
            h=FORECAST_HORIZON,
            input_size=504,
            patch_len=16,
            stride=8,
            hidden_size=128,
            n_heads=8,
            learning_rate=1e-3,
            max_steps=500,
            batch_size=32
        )
    
    if BEST_MODEL == "TFT":
        model = TFT(
            h=FORECAST_HORIZON,
            input_size=252,
            hidden_size=64,
            n_head=4,
            learning_rate=1e-3,
            max_steps=100,
            batch_size=32,
            hist_exog_list=[
                "volume",
                "sma20",
                "sma50",
                "ema20",
                "rsi",
                "macd",
                "macd_signal",
                "bb_low",
                "bb_high",
                "returns",
                "volatility",
            ]
        )
        
    if BEST_MODEL == "NBEATSx":
         model = NBEATSx(
            h=FORECAST_HORIZON,
            input_size=252,
        
            hist_exog_list=[
                "volume",
                "sma20",
                "sma50",
                "ema20",
                "rsi",
                "macd",
                "macd_signal",
                "bb_low",
                "bb_high",
                "returns",
                "volatility",
            ],
        
            learning_rate=1e-3,
            max_steps=300,
            batch_size=32,
        )

    
    nf = NeuralForecast(
        models=[model],
        freq='D'
    )
    
    # TRAIN ALL HISTORICAL DATA
    nf.fit(df=nf_df)
    
    future_forecast = nf.predict()
    forecast_series = future_forecast[BEST_MODEL]

    # Buy Point
    lowest_idx = forecast_series.idxmin()
    lowest_price = forecast_series.loc[lowest_idx]
    lowest_horizon = lowest_idx + 1
    
    
    future_after_buy = forecast_series.loc[lowest_idx:]
    
    # Sell
    highest_idx = future_after_buy.idxmax()
    highest_price = future_after_buy.loc[highest_idx]
    highest_horizon = highest_idx + 1

    # Expected return
    expected_return = (
        (highest_price - lowest_price)
        / lowest_price
    ) * 100
    volatility = df["volatility"].tail(60).mean()
    
   
    results.append({
        "Ticker": ticker,
        "HighestPrice": highest_price,
        "HighestHorizon": f"H+{highest_horizon}",
        "LowestPrice": lowest_price,
        "LowestHorizon": f"H+{lowest_horizon}",
        "ExpectedReturn": expected_return,
        "Volatility": volatility,
    })

Seed set to 1
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name         | Type              | Params | Mode 
-----------------------------------------------------------
0 | loss         | MAE               | 0      | train
1 | padder_train | ConstantPad1d     | 0      | train
2 | scaler       | TemporalNorm      | 0      | train
3 | model        | PatchTST_backbone | 1.4 M  | train
-----------------------------------------------------------
1.4 M     Trainable params
3         Non-trainable params
1.4 M     Total params
5.502     Total estimated model params size (MB)
90        Modules in train mode
0         Modules in eval mode


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_steps=500` reached.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Predicting: |          | 0/? [00:00<?, ?it/s]

## Calculate Profit by Top 5 Expected Return

In [27]:
EXPORT_TOP5_FORECAST = f"./reports/top5_forecast_{BEST_MODEL}.csv"

In [28]:
import pandas as pd
import numpy as np

result_df = pd.DataFrame(results)

result_df = result_df[result_df["ExpectedReturn"] > 0].copy()
result_df["Volatility"] = result_df["Volatility"].replace(0, np.nan)
result_df["Score"] = (result_df["ExpectedReturn"] / result_df["Volatility"])

result_df = result_df.sort_values(
    by="Score",
    ascending=False
)

top5 = result_df.head(5).copy()
top5["Weight"] = (
    top5["Score"]
    / top5["Score"].sum()
)

TOTAL_FUNDS = 1_000_000_000_000
top5["Allocation"] = (
    top5["Weight"]
    * TOTAL_FUNDS
)
top5["ProjectedProfit"] = (
    top5["Allocation"]
    * top5["ExpectedReturn"]
    / 100
)

# Export
top5.to_csv(
    EXPORT_TOP5_FORECAST,
    index=False
)

# Total Profit
total_profit = top5["ProjectedProfit"].sum()
print(top5[[
    "Ticker",
    "ExpectedReturn",
    "Volatility",
    "Score",
    "Weight",
    "Allocation",
    "ProjectedProfit"
]])

print("\n====================")
print("TOTAL PROFIT")
print("====================")
print(f"Rp {total_profit:,.0f}")

  Ticker  ExpectedReturn  Volatility        Score    Weight    Allocation  \
3   AAPL       36.872318    0.014350  2569.470231  0.316400  3.164002e+11   
1    AXP       29.953354    0.017544  1707.338564  0.210239  2.102388e+11   
5     KO       15.315552    0.010595  1445.510575  0.177998  1.779977e+11   
0    BAC       18.695518    0.014531  1286.551292  0.158424  1.584237e+11   
6    CVX       17.727932    0.015941  1112.079616  0.136940  1.369396e+11   

   ProjectedProfit  
3     1.166641e+11  
1     6.297356e+10  
5     2.726133e+10  
0     2.961814e+10  
6     2.427656e+10  

TOTAL PROFIT
Rp 260,793,676,091
